#  Notebook 01 — Data Loading & Initial Exploration
## Spotify Music Analytics Dataset (2015–2025)

**Goal:** Load the raw CSV, understand its shape, types, missing values, and distributions — before touching anything.

> This is the *discovery* phase. We never modify data here — only observe.


In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ── Load the raw dataset ──────────────────────────────────────────────────────
RAW_PATH = '../data/raw/spotify_2015_2025_85k.csv'
df = pd.read_csv(RAW_PATH)

print(f" Dataset loaded successfully")
print(f"   Rows    : {df.shape[0]:,}")
print(f"   Columns : {df.shape[1]}")


## 1️⃣ Shape & Column Names
First, we check how big the dataset is and what columns exist. This tells us whether we have the right file and gives us a mental map of the data.


In [ ]:
print("Shape:", df.shape)
print("\nColumn names:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:2d}. {col}")


## 2️⃣ Data Types & Memory Usage
Understanding data types is critical — wrong types waste memory and cause bugs during analysis (e.g. dates stored as strings can't be filtered by year).


In [ ]:
print("Data types:\n")
print(df.dtypes.to_string())
print(f"\nTotal memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")


## 3️⃣ Descriptive Statistics — Numerical Columns
`describe()` gives us the min, max, mean, standard deviation, and quartiles for every numerical column. We use this to spot impossible values (e.g. negative duration) and skewed distributions.


In [ ]:
df.describe().T.style.format("{:.3f}").background_gradient(cmap='Blues', axis=1)


## 4️⃣ Descriptive Statistics — Categorical Columns
For text columns, we care about how many unique values exist and what the most frequent ones are.


In [ ]:
df.describe(include='object')


## 5️⃣ Missing Values
Missing data can silently distort analysis. Here we count nulls per column and express them as a percentage of total rows, so we can decide whether to fill or drop.


In [ ]:
missing = pd.DataFrame({
    'missing_count': df.isnull().sum(),
    'missing_%': (df.isnull().sum() / len(df) * 100).round(2)
}).sort_values('missing_%', ascending=False)

print("Missing values per column:")
print(missing[missing['missing_count'] > 0].to_string())
print(f"\nTotal missing cells: {df.isnull().sum().sum()}")


## 6️⃣ Duplicate Check
Duplicates inflate counts and skew averages. We check for fully duplicate rows (same values in every column) here.


In [ ]:
dupes = df.duplicated().sum()
print(f"Fully duplicate rows: {dupes:,}")
print(f"Duplicate rate      : {dupes / len(df) * 100:.2f}%")


## 7️⃣ Sample Rows
Looking at the first and last 5 rows gives us a sanity check that the data was loaded correctly and helps spot obvious formatting issues.


In [ ]:
print("── First 5 rows ──────────────────────────────────")
display(df.head())
print("\n── Last 5 rows ───────────────────────────────────")
display(df.tail())


## 8️⃣ Unique Value Counts & Categorical Distributions
Knowing how many unique values each column has tells us cardinality — important for deciding how to encode or group categories later.


In [ ]:
print("Unique value counts per column:")
unique_counts = df.nunique().sort_values(ascending=False)
for col, n in unique_counts.items():
    print(f"  {col:<25} {n:>7,} unique values")

print("\n── Top 10 value_counts for categorical columns ──")
for col in df.select_dtypes(include='object').columns:
    if df[col].nunique() < 500:   # skip high-cardinality cols like track_id
        print(f"\n{col}:")
        print(df[col].value_counts().head(10).to_string())


##  Summary of Findings

| Topic | Finding |
|-------|---------|
| **Rows / Columns** | 85,000 rows × 19 columns |
| **Missing values** | Only `track_name` (21) and `album_name` (46) — <0.1% each |
| **Duplicates** | 0 fully duplicate rows |
| **Data type issues** | `release_date` is object (string) — needs conversion to datetime; `mode` and `key` are int but are actually categorical |
| **Columns to clean** | `release_date`, `track_name`, `album_name` |
| **Columns to engineer** | `release_year`, `release_month`, `duration_min`, `popularity_tier`, `energy_level`, `is_danceable`, `stream_millions` |
| **Key ranges** | popularity 0–100, danceability 0–1, energy 0–1, tempo 50–200 BPM |
| **Genres** | 12 genres, very evenly distributed (~7,000 tracks each) |
| **Countries** | 10+ countries, India and Canada are most represented |

**Next step → Notebook 02: Data Cleaning**
